In [ ]:
# Final MaxViT + mBART + PAL training script (compact, robust)

import os, json, random, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

from transformers import (
    AutoModelForSeq2SeqLM,
    MBart50TokenizerFast,
    get_linear_schedule_with_warmup,
)
from transformers.modeling_outputs import BaseModelOutput

# -----------------------------
# Device + perf
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -----------------------------
# Paths (edit as needed)
# -----------------------------
DATA_DIR = "/kaggle/input/synthetic-bic-pairs/synthetic-bengali-images"
REAL_IMAGE_BASE_DIR = "/kaggle/input/coco-image-caption/train2014/train2014"
COCO_ANNOTATIONS_PATH = '/kaggle/input/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json'

MODEL_CHECKPOINT_PATH = "/kaggle/working/maxvit_mbart_bn_checkpoint.pt"
MODEL_CHECKPOINT_PATH_IN = "/kaggle/input/train/transformers/default/16/maxvit_mbart_bn_checkpoint.pt"
MODEL_WEIGHTS_OUT = "/kaggle/working/maxvit_mbart_bn_weights.pth"

# -----------------------------
# Models + Train cfg
# -----------------------------
MAXVIT_MODEL_NAME = "maxvit_base_tf_224.in1k"
MBART_MODEL_NAME  = "facebook/mbart-large-50-many-to-many-mmt"

BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_CAPTION_LENGTH = 96

# PAL schedule
BASE_PAL_WEIGHT = 0.20
PAL_WARMUP_STEPS = 2000

PAL_CFG = {
    "attn_temp": 0.8,                 # sharper weighting
    "topk_ratio": 1.0,                # no pruning early; we reduce later in training
    "use_last_k_cross_layers": 4,
    "pool_factor": 2,
    "multi_scale": True,
}

USE_INFO_NCE = False
INFO_NCE_CFG = {"temp": 0.07, "coef": 0.3}

USE_OT = False
OT_CFG = {"reg": 0.05, "iters": 30, "topk_ratio": 0.10, "coef": 0.5}

# -----------------------------
# Helpers
# -----------------------------
def safe_load_image(img_path: str):
    try:
        img = Image.open(img_path).convert('RGB')
        return img
    except (FileNotFoundError, UnidentifiedImageError) as e:
        print(f"[safe_load_image] {e} @ {img_path}")
        return None
    except Exception as e:
        print(f"[safe_load_image] Unexpected: {e} @ {img_path}")
        return None

def extract_captions(full_caption: str):
    # Expect "A photo of: EN. In Bengali: BN"
    if "In Bengali:" in full_caption:
        caption_en = full_caption.split("In Bengali:")[0].strip()
        caption_bn = full_caption.split("In Bengali:")[-1].strip()
    else:
        parts = full_caption.strip().split(". ")
        if len(parts) >= 2:
            caption_en = parts[0].strip()
            caption_bn = parts[-1].strip()
        else:
            caption_en = full_caption.strip()
            caption_bn = full_caption.strip()
    if caption_en.startswith("A photo of: "):
        caption_en = caption_en[len("A photo of: "):].strip()
    return caption_en, caption_bn

def handle_full_stop_variation(caption_en):
    caps = [caption_en]
    if caption_en.endswith('.'):
        caps.append(caption_en[:-1].strip())
    else:
        caps.append(caption_en + '.')
    return caps

def clean_bengali_caption(caption_text: str):
    import unicodedata
    cleaned = unicodedata.normalize('NFKC', str(caption_text))
    cleaned = (cleaned.replace('‘', "'").replace('’', "'")
                      .replace('“', '"').replace('”', '"')
                      .replace('—', '-').replace('–', '-')
                      .replace('…', '...'))
    cleaned = ''.join(ch for ch in cleaned if ch.isprintable())
    cleaned = ' '.join(cleaned.split()).strip()
    cleaned = cleaned.encode('utf-8', 'ignore').decode('utf-8', 'ignore')
    return cleaned

In [ ]:
# -----------------------------
# Dataset (real + synthetic)
# -----------------------------
class BengaliCaptionDataset(Dataset):
    def __init__(self, df, image_transform, tokenizer, max_length=MAX_CAPTION_LENGTH, p_syn=0.5):
        self.df = df.reset_index(drop=True)
        self.tx = image_transform
        self.tok = tokenizer
        self.max_length = int(max_length)
        self.pad_id = int(self.tok.pad_token_id)
        self.p_syn = float(p_syn)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Always return both real and synthetic; decoder CE uses real, PAL uses both
        real_path = row["real_image_path"]
        gen_path  = row["generated_image_path"]
        cap_bn    = clean_bengali_caption(row["caption_bn"])

        real_img = safe_load_image(real_path); gen_img = safe_load_image(gen_path)
        if real_img is None or gen_img is None:
            return None

        try:
            real_pixel_values = self.tx(real_img)
            synth_pixel_values = self.tx(gen_img)
        except Exception:
            return None

        toks = self.tok(
            cap_bn,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding="max_length"
        )
        labels = toks.input_ids.squeeze(0)
        attn   = toks["attention_mask"].squeeze(0)

        # mask PAD tokens
        labels[attn == 0] = -100
        if torch.all(labels == -100):
            return None

        return {
            "real_pixel_values": real_pixel_values,
            "synthetic_pixel_values": synth_pixel_values,
            "labels": labels,
            "attention_mask": attn
        }

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    return {
        "real_pixel_values": torch.stack([b["real_pixel_values"] for b in batch], dim=0),
        "synthetic_pixel_values": torch.stack([b["synthetic_pixel_values"] for b in batch], dim=0),
        "labels": torch.stack([b["labels"] for b in batch], dim=0),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch], dim=0),
    }

def preflight_filter(df, tokenizer, image_transform, max_length=MAX_CAPTION_LENGTH, limit=None):
    keep = []
    for i, row in enumerate(df.itertuples(index=False)):
        if limit and i >= limit: break
        ok_paths = os.path.exists(row.real_image_path) and os.path.exists(row.generated_image_path)
        if not ok_paths: keep.append(False); continue
        img1 = safe_load_image(row.real_image_path); img2 = safe_load_image(row.generated_image_path)
        if img1 is None or img2 is None: keep.append(False); continue
        try:
            _ = image_transform(img1); _ = image_transform(img2)
            _ = tokenizer(clean_bengali_caption(row.caption_bn),
                          return_tensors="pt", truncation=True, max_length=max_length, padding="max_length")
            keep.append(True)
        except Exception:
            keep.append(False)
    return pd.Series(keep).values


In [ ]:
# -----------------------------
# Core model (MaxViT + mBART + PAL)
# -----------------------------
def _l2n(x, dim=-1, eps=1e-8):
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))

class MaxVitMbartCaptioningModel(nn.Module):
    def __init__(
        self,
        encoder_model_name,
        decoder_model_name,
        bn_in_token_id,
        patch_alignment_weight=0.05,
        attn_temp: float = 1.0,
        topk_ratio: float = 1.0,
        use_last_k_cross_layers: int = 2,
        pool_factor: int = 1,
        use_info_nce: bool = False,
        info_nce_temp: float = 0.07,
        info_nce_coef: float = 0.3,
        use_ot: bool = False,
        ot_reg: float = 0.05,
        ot_iters: int = 50,
        ot_topk_ratio: float = 0.15,
        ot_coef: float = 0.5,
        multi_scale: bool = False
    ):
        super().__init__()
        # Vision encoder (frozen)
        self.vision_encoder = timm.create_model(
            encoder_model_name, pretrained=True, features_only=True
        )
        self.vision_encoder.eval()
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        if hasattr(self.vision_encoder, "set_grad_checkpointing"):
            self.vision_encoder.set_grad_checkpointing(True)

        ch_list = self.vision_encoder.feature_info.channels()
        self.last_ch = ch_list[-1]
        self.prev_ch = ch_list[-2] if len(ch_list) >= 2 else None

        # Language decoder
        self.language_decoder = AutoModelForSeq2SeqLM.from_pretrained(
            decoder_model_name, attn_implementation="eager", low_cpu_mem_usage=True
        )
        self.language_decoder.config.decoder_start_token_id = bn_in_token_id
        self.language_decoder.config.use_cache = False
        if hasattr(self.language_decoder, "gradient_checkpointing_enable"):
            self.language_decoder.gradient_checkpointing_enable()

        self.d_model = self.language_decoder.config.d_model
        self.dec_dtype = next(self.language_decoder.parameters()).dtype

        # Vision bridge (2-layer MLP) + LN
        self.vision_bridge = nn.Sequential(
            nn.Linear(self.last_ch, self.d_model),
            nn.GELU(),
            nn.Linear(self.d_model, self.d_model),
        ).to(self.dec_dtype)
        self.encoder_norm = nn.LayerNorm(self.d_model, eps=1e-5).to(self.dec_dtype)
        nn.init.xavier_uniform_(self.vision_bridge[0].weight); nn.init.zeros_(self.vision_bridge[0].bias)
        nn.init.xavier_uniform_(self.vision_bridge[2].weight); nn.init.zeros_(self.vision_bridge[2].bias)

        # optional prev-scale head
        self.multi_scale = bool(multi_scale and self.prev_ch is not None)
        if self.multi_scale:
            self.vision_projection_prev = nn.Linear(self.prev_ch, self.d_model).to(self.dec_dtype)
            nn.init.xavier_uniform_(self.vision_projection_prev.weight)
            if self.vision_projection_prev.bias is not None:
                nn.init.zeros_(self.vision_projection_prev.bias)

        # tiny 2D positional MLP
        self.pos_mlp = nn.Sequential(
            nn.Linear(8, self.d_model), nn.GELU(), nn.Linear(self.d_model, self.d_model)
        ).to(self.dec_dtype)

        # knobs
        self.patch_alignment_weight = float(patch_alignment_weight)
        self.attn_temp = float(attn_temp)
        self.topk_ratio = float(topk_ratio)
        self.use_last_k_cross_layers = int(max(1, use_last_k_cross_layers))
        self.pool_factor = int(max(1, pool_factor))
        self.use_info_nce = bool(use_info_nce)
        self.info_nce_temp = float(info_nce_temp)
        self.info_nce_coef = float(info_nce_coef)
        self.use_ot = bool(use_ot)
        self.ot_reg = float(ot_reg)
        self.ot_iters = int(ot_iters)
        self.ot_topk_ratio = float(ot_topk_ratio)
        self.ot_coef = float(ot_coef)

    def set_trainable_decoder_layers(self, last_k: int):
        # Freeze all first
        for p in self.language_decoder.parameters():
            p.requires_grad = False
        # Unfreeze top-k decoder layers + lm_head
        dec_layers = self.language_decoder.model.decoder.layers
        last_k = max(0, min(last_k, len(dec_layers)))
        for l in range(len(dec_layers) - last_k, len(dec_layers)):
            for p in dec_layers[l].parameters():
                p.requires_grad = True
        for p in self.language_decoder.lm_head.parameters():
            p.requires_grad = True

    def _make_pos(self, H, W, device, dtype):
        ys, xs = torch.meshgrid(
            torch.linspace(-1, 1, H, device=device, dtype=dtype),
            torch.linspace(-1, 1, W, device=device, dtype=dtype),
            indexing="ij"
        )
        r = torch.sqrt(xs**2 + ys**2)
        pos = torch.stack(
            [xs, ys, xs*ys, xs**2, ys**2, r, torch.sin(xs), torch.cos(ys)], dim=-1
        ).view(H*W, 8)
        return pos

    # ---------- features ----------
    def _encode_feats_list(self, pixel_values):
        with torch.no_grad():
            feats_list = self.vision_encoder(pixel_values)
        outs = []
        idxs = [-1] if not self.multi_scale else [-2, -1]
        for i in idxs:
            f = feats_list[i]
            if self.pool_factor > 1:
                H, W = f.shape[-2:]
                f = F.adaptive_avg_pool2d(
                    f, (max(1, H // self.pool_factor), max(1, W // self.pool_factor))
                )
            outs.append(f)
        return outs

    def _project_norm(self, fmap, use_prev: bool = False):
        B, C, H, W = fmap.shape
        S = H * W
        patches = fmap.reshape(B, C, S).transpose(1, 2)
        proj = self.vision_bridge if not use_prev else self.vision_projection_prev
        # cast to proj dtype
        if isinstance(proj, nn.Sequential):
            patches = patches.to(dtype=proj[0].weight.dtype)
        else:
            patches = patches.to(dtype=proj.weight.dtype)
        enc = proj(patches)  # (B,S,D)
        # add 2D positional signal
        pos = self._make_pos(H, W, enc.device, enc.dtype)  # (S,8)->(S,D)
        pos = self.pos_mlp(pos)
        enc = enc + pos.unsqueeze(0)
        enc = self.encoder_norm(enc.to(self.encoder_norm.weight.dtype))
        return enc, (H, W)

    # ---------- attn -> patch weights ----------
    def _attention_to_patch_weights(self, cross_attentions, attention_mask, S):
        keep = cross_attentions[-self.use_last_k_cross_layers:]
        attn = [torch.nan_to_num(a.float(), 0.0, 0.0, 0.0) for a in keep if a is not None]
        if not attn:
            B = attention_mask.size(0) if attention_mask is not None else 1
            device = attention_mask.device if attention_mask is not None else next(self.language_decoder.parameters()).device
            return torch.full((B, S), 1.0 / S, device=device)

        cross = torch.stack(attn, dim=0)  # (L,B,H,T,S)
        if attention_mask is not None:
            T = cross.size(3)
            tgt = attention_mask[:, :T].to(cross.dtype).view(1, -1, 1, T, 1)
            cross = (cross * tgt).sum(dim=3, keepdim=True) / tgt.sum(dim=3, keepdim=True).clamp_min(1.0)
            cross = cross.squeeze(3)
        else:
            cross = cross.mean(dim=3)
        w = cross.mean(dim=2).mean(dim=0)            # (B,S)
        w = torch.softmax(w / max(1e-6, self.attn_temp), dim=-1)

        if 0.0 < self.topk_ratio < 1.0:
            k = max(1, min(int(self.topk_ratio * w.size(1)), w.size(1)))
            topv, topi = torch.topk(w, k, dim=1)
            mask = torch.zeros_like(w).scatter_(1, topi, 1.0)
            w = w * mask
            w = w / w.sum(dim=1, keepdim=True).clamp_min(1e-8)

        bad = ~torch.isfinite(w).all(dim=1, keepdim=True)
        if bad.any():
            uniform = torch.full_like(w, 1.0 / w.size(1))
            w = torch.where(bad, uniform, w)
        return w

    # ---------- OT helper ----------
    def _sinkhorn_cost(self, r, s, wr, ws, eps=0.05, iters=50):
        K = torch.exp((r @ s.t() - 1.0) / eps)
        u = torch.ones_like(wr) / wr.numel()
        v = torch.ones_like(ws) / ws.numel()
        for _ in range(iters):
            u = wr / (K @ v).clamp_min(1e-12)
            v = ws / (K.t() @ u).clamp_min(1e-12)
        T = torch.diag(u) @ K @ torch.diag(v)
        C = 1.0 - (r @ s.t())
        return (T * C).sum()

    # ---------- forward ----------
    def forward(self, real_pixel_values, synthetic_pixel_values, labels=None, attention_mask=None):
        real_feats_list  = self._encode_feats_list(real_pixel_values)
        synth_feats_list = self._encode_feats_list(synthetic_pixel_values)

        if self.multi_scale:
            real_prev, real_last   = real_feats_list
            synth_prev, synth_last = synth_feats_list
        else:
            real_last   = real_feats_list[-1]
            synth_last  = synth_feats_list[-1]

        real_proj_last, (H_l, W_l) = self._project_norm(real_last, use_prev=False)
        synth_proj_last, _          = self._project_norm(synth_last, use_prev=False)
        S_last = H_l * W_l

        # decoder CE
        real_proj_last = real_proj_last.to(self.dec_dtype)
        enc_out = BaseModelOutput(last_hidden_state=real_proj_last)

        # explicit all-ones encoder mask
        enc_mask = torch.ones((real_proj_last.size(0), real_proj_last.size(1)),
                              dtype=torch.long, device=real_proj_last.device)

        dec_out = self.language_decoder(
            encoder_outputs=enc_out,
            labels=labels,
            decoder_attention_mask=attention_mask,  # this is the text-side mask, keep it
            output_attentions=True,
            return_dict=True
        )
        ce = dec_out.loss

        # PAL weights from cross-attn
        with torch.no_grad():
            w_last = self._attention_to_patch_weights(dec_out.cross_attentions, attention_mask, S_last)

        with torch.amp.autocast('cuda', enabled=False):
            r = real_proj_last.float()
            s = synth_proj_last.float()
            w = w_last.float().unsqueeze(-1)
            r_pool = (r * w).sum(dim=1)
            s_pool = (s * w).sum(dim=1)
            sim = F.cosine_similarity(r_pool, s_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
            pal_last = (1.0 - sim).mean()
            total_pal = pal_last

            if self.multi_scale:
                real_proj_prev, (H_p, W_p) = self._project_norm(real_prev, use_prev=True)
                synth_proj_prev, _ = self._project_norm(synth_prev, use_prev=True)
                w_grid = w_last.view(w_last.size(0), 1, H_l, W_l)
                w_prev = F.interpolate(w_grid, size=(H_p, W_p), mode='bilinear', align_corners=False).flatten(2).squeeze(1)
                w_prev = w_prev / w_prev.sum(dim=1, keepdim=True).clamp_min(1e-8)
                rp = real_proj_prev.float(); sp = synth_proj_prev.float(); wp = w_prev.float().unsqueeze(-1)
                rp_pool = (rp * wp).sum(dim=1); sp_pool = (sp * wp).sum(dim=1)
                sim_p = F.cosine_similarity(rp_pool, sp_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
                pal_prev = (1.0 - sim_p).mean()
                total_pal = 0.5 * (pal_last + pal_prev)

            nce = r_pool.new_tensor(0.0)
            if self.use_info_nce and r_pool.size(0) >= 2:
                r_n = _l2n(r_pool); s_n = _l2n(s_pool)
                feats = torch.cat([r_n, s_n], dim=0)
                logits = (feats @ feats.t()) / max(1e-6, self.info_nce_temp)
                logits.fill_diagonal_(-1e9)
                idx = torch.arange(feats.size(0), device=feats.device)
                pos = idx ^ (feats.size(0)//2)
                nce = -F.log_softmax(logits, dim=1)[idx, pos].mean()

            ot_cost = r_pool.new_tensor(0.0)
            if self.use_ot:
                B, S = r.size(0), r.size(1)
                if 0.0 < self.ot_topk_ratio < 1.0:
                    k = max(1, min(int(self.ot_topk_ratio * S), S))
                    wv, wi = torch.topk(w_last, k, dim=1)
                else:
                    wi = torch.arange(S, device=r.device).unsqueeze(0).expand(B, -1)
                    wv = w_last
                for b in range(B):
                    ridx = wi[b]; sidx = wi[b]
                    rnorm = _l2n(r[b, ridx, :], dim=-1); snorm = _l2n(s[b, sidx, :], dim=-1)
                    wr = (wv[b] / wv[b].sum()).detach(); ws = (wv[b] / wv[b].sum()).detach()
                    ot_cost = ot_cost + self._sinkhorn_cost(rnorm, snorm, wr, ws, eps=self.ot_reg, iters=self.ot_iters)
                ot_cost = ot_cost / B

        total = ce + self.patch_alignment_weight * total_pal
        if self.use_info_nce:
            total = total + (self.info_nce_coef * self.patch_alignment_weight) * nce
        if self.use_ot:
            total = total + (self.ot_coef * self.patch_alignment_weight) * ot_cost

        return {
            "loss": total,
            "cross_entropy_loss": ce,
            "patch_alignment_loss": total_pal,
            "info_nce_loss": (nce if self.use_info_nce else None),
            "ot_loss": (ot_cost if self.use_ot else None),
            "logits": dec_out.logits
        }

    @torch.no_grad()
    def generate(self, real_pixel_values, tokenizer, max_new_tokens=128, num_beams=6,
                 num_return_sequences=None, length_penalty=1.3, no_repeat_ngram_size=3,
                 min_new_tokens=16, repetition_penalty=1.02):
        self.eval()
        last = self._encode_feats_list(real_pixel_values)[-1]
        enc_last, _ = self._project_norm(last, use_prev=False)
        enc_last = enc_last.to(dtype=self.dec_dtype)
        encoder_outputs = BaseModelOutput(last_hidden_state=enc_last)
        bos_id = tokenizer.lang_code_to_id.get("bn_IN")
        if bos_id is None:
            raise ValueError("bn_IN not found in tokenizer.lang_code_to_id")
        eos_id = tokenizer.eos_token_id; pad_id = tokenizer.pad_token_id
        if num_return_sequences is None:
            num_return_sequences = num_beams
        prev_cache = getattr(self.language_decoder.config, "use_cache", True)
        self.language_decoder.config.use_cache = True
        try:
            out = self.language_decoder.generate(
                encoder_outputs=encoder_outputs,
                decoder_start_token_id=bos_id,
                forced_bos_token_id=bos_id,
                eos_token_id=eos_id,
                pad_token_id=pad_id,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                length_penalty=max(1.0, float(length_penalty)),
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                early_stopping=True,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                return_dict_in_generate=True
            )
        finally:
            self.language_decoder.config.use_cache = prev_cache
        return tokenizer.batch_decode(out.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True)


In [ ]:
# -----------------------------
# Checkpoint helpers
# -----------------------------
def save_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = {
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
    }
    torch.save(ckpt, path)
    print(f"Saved checkpoint to {path}")

def load_checkpoint_if_any(path, device, model, optimizer, scheduler, scaler):
    if not os.path.exists(path):
        print("No previous checkpoint found, training from scratch.")
        return
    ckpt = torch.load(path, map_location=device)
    if "model_state_dict" in ckpt:
        missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print(f"Model loaded. Missing: {len(missing)} | Unexpected: {len(unexpected)}")
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("Optimizer/Scheduler/Scaler states loaded.")
        except Exception as e:
            print(f"Could not load optimizer/scheduler/scaler states: {e} (reinit)")
    else:
        model.load_state_dict(ckpt, strict=False)
        print("Legacy weights loaded.")

# -----------------------------
# Train (7 epochs) with warmups
# -----------------------------
def train_for_seven_epochs(
    model, dataloader, optimizer, scheduler, scaler, device,
    grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=7
):
    # Start with top-2 decoder layers trainable
    model.set_trainable_decoder_layers(last_k=2)

    global_step = 0
    for epoch in range(num_epochs):
        # Progressive unfreeze
        if epoch == 2:
            model.set_trainable_decoder_layers(last_k=6)
        if epoch == 4:
            model.set_trainable_decoder_layers(last_k=12)  # full decoder

        # Late introduce sparsity
        if epoch >= 3:
            model.topk_ratio = 0.5
        if epoch >= 5:
            model.topk_ratio = 0.25

        model.train()
        total_epoch_loss = 0.0
        steps = 0
        optimizer.zero_grad(set_to_none=True)

        for batch in DataLoader(dataloader.dataset, batch_size=dataloader.batch_size, shuffle=True,
                                num_workers=0, pin_memory=True, collate_fn=collate_fn):
            if batch is None: continue

            # PAL warmup
            pal_scale = min(1.0, global_step / PAL_WARMUP_STEPS)
            model.patch_alignment_weight = BASE_PAL_WEIGHT * pal_scale

            real_pixel_values      = batch["real_pixel_values"].to(device, non_blocking=True)
            synthetic_pixel_values = batch["synthetic_pixel_values"].to(device, non_blocking=True)
            labels                 = batch["labels"].to(device, non_blocking=True)
            decoder_attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            # label sanity (first few steps)
            if global_step < 5:
                assert (labels != -100).any(), "All labels masked; check tokenizer/max_length."

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device=="cuda")):
                outputs = model(
                    real_pixel_values=real_pixel_values,
                    synthetic_pixel_values=synthetic_pixel_values,
                    labels=labels,
                    attention_mask=decoder_attention_mask
                )
                loss = outputs["loss"] / grad_accum_steps

            scaler.scale(loss).backward()
            if (steps + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            total_epoch_loss += float(outputs["loss"].detach())
            steps += 1
            global_step += 1

            del real_pixel_values, synthetic_pixel_values, labels, decoder_attention_mask, outputs
            torch.cuda.empty_cache()

        print(f"Epoch {epoch+1}: Avg Total Loss = {total_epoch_loss/max(1,steps):.4f}")
        torch.cuda.empty_cache()

# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":
    print("--- Starting Data Preprocessing ---")

    # 1) Load COCO EN captions to map -> image_id
    with open(COCO_ANNOTATIONS_PATH, 'r') as f:
        coco_data = json.load(f)
    annotations = coco_data['annotations']
    df_mscoco = pd.DataFrame(annotations)[['id', 'image_id', 'caption']]
    df_mscoco.rename(columns={'id': 'caption_id', 'caption': 'caption_en'}, inplace=True)

    # 2) Scan synthetic JSONs (NO ARTIFICIAL LIMIT)
    results = []
    json_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".json"))[:800]
    print(f"Scanning JSON files: {len(json_files)}")
    for j in json_files:
        try:
            with open(os.path.join(DATA_DIR, j), encoding="utf-8") as f:
                meta = json.load(f)
            full_caption = meta["caption_bn"]
            caption_en, caption_bn = extract_captions(full_caption)
            variants = handle_full_stop_variation(caption_en)

            matched_rows = None
            for v in variants:
                matched_rows = df_mscoco[df_mscoco['caption_en'] == v]
                if not matched_rows.empty: break
            if matched_rows is None or matched_rows.empty:
                continue

            image_id = matched_rows['image_id'].values[0]
            image_id_str = str(image_id).zfill(12)
            real_image_path = os.path.join(REAL_IMAGE_BASE_DIR, f"COCO_train2014_{image_id_str}.jpg")
            generated_image_path = os.path.join(DATA_DIR, meta["filename"])

            if os.path.exists(real_image_path) and os.path.exists(generated_image_path):
                results.append({
                    "real_image_path": real_image_path,
                    "generated_image_path": generated_image_path,
                    "caption_bn": caption_bn,
                    "caption_en": caption_en,
                    "valid": True
                })
        except Exception:
            pass

    df_results = pd.DataFrame(results)
    print(f"Found {len(df_results)} valid pairs.")
    if df_results.empty:
        raise SystemExit("No valid data pairs found. Check paths/structure.")

    # 3) Tokenizer
    tokenizer = MBart50TokenizerFast.from_pretrained(
        MBART_MODEL_NAME, src_lang="bn_IN", tgt_lang="bn_IN"
    )
    bn_in_token_id = tokenizer.lang_code_to_id.get("bn_IN")

    # 4) Build timm-native transforms for MaxViT
    # Create a temporary model to get its data config
    _tmp_model = timm.create_model(MAXVIT_MODEL_NAME, pretrained=True)
    cfg = resolve_data_config({}, model=_tmp_model)
    del _tmp_model
    train_transform_safe = create_transform(**cfg, is_training=True)
    eval_transform_safe  = create_transform(**cfg, is_training=False)

    # 5) Preflight filter
    keep_idx = preflight_filter(
        df_results, tokenizer, image_transform=train_transform_safe,
        max_length=MAX_CAPTION_LENGTH
    )
    df_results = df_results.loc[keep_idx].reset_index(drop=True)
    print(f"After preflight: {len(df_results)} usable pairs.")

    # 6) Split
    if len(df_results) > 10:
        df_train = df_results.iloc[:-10].reset_index(drop=True)
        df_val   = df_results.iloc[-10:].reset_index(drop=True)
    else:
        df_train = df_results.copy()
        df_val   = pd.DataFrame(columns=df_results.columns)

    # 7) DataLoaders
    ds_train = BengaliCaptionDataset(df_train, train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          pin_memory=True, collate_fn=collate_fn)

    # 8) Build model
    model = MaxVitMbartCaptioningModel(
        MAXVIT_MODEL_NAME, MBART_MODEL_NAME,
        bn_in_token_id=bn_in_token_id,
        patch_alignment_weight=0.0,  # will be warmed up dynamically
        attn_temp=PAL_CFG["attn_temp"],
        topk_ratio=PAL_CFG["topk_ratio"],
        use_last_k_cross_layers=PAL_CFG["use_last_k_cross_layers"],
        pool_factor=PAL_CFG["pool_factor"],
        multi_scale=PAL_CFG["multi_scale"],
        use_info_nce=USE_INFO_NCE,
        info_nce_temp=INFO_NCE_CFG["temp"],
        info_nce_coef=INFO_NCE_CFG["coef"],
        use_ot=USE_OT,
        ot_reg=OT_CFG["reg"],
        ot_iters=OT_CFG["iters"],
        ot_topk_ratio=OT_CFG["topk_ratio"],
        ot_coef=OT_CFG["coef"]
    ).to(DEVICE)

    # 9) Optim, scheduler, scaler (linear warmup/decay)
    t_total = max(1, len(dl_train)) * NUM_EPOCHS // max(1, GRAD_ACCUM_STEPS)
    warmup_steps = int(0.06 * t_total)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, t_total)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE=="cuda"))

    # 10) Optional resume
    if os.path.exists(MODEL_CHECKPOINT_PATH_IN):
        load_checkpoint_if_any(MODEL_CHECKPOINT_PATH_IN, DEVICE, model, optimizer, scheduler, scaler)

    # 11) Train
    train_for_seven_epochs(
        model=model, dataloader=dl_train,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler,
        device=DEVICE, grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=NUM_EPOCHS
    )

    # 12) Save
    save_checkpoint(MODEL_CHECKPOINT_PATH, model, optimizer, scheduler, scaler)
    torch.save(model.state_dict(), MODEL_WEIGHTS_OUT)
    print(f"✅ Saved model weights to {MODEL_WEIGHTS_OUT}")

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/salaniz/pycocoevalcap.git

# Step 2: Install it
!cd pycocoevalcap && python setup.py install

# Step 3: Add it to sys.path so Python can find it immediately
import sys
sys.path.append("/kaggle/working/pycocoevalcap")

!pip install -q bert-score

In [ ]:
# ================================
# PAL Ablation: patch_weight × attn_temp × topk_ratio
# ================================
import itertools, math, time, datetime, gc, random, numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import torch
from torch.utils.data import DataLoader
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# ---- Safety: optional metric deps ----
HAVE_COCOCAP = True
try:
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.spice.spice import Spice
except Exception as _e:
    print("pycocoevalcap not fully available; METEOR/ROUGE/CIDEr/SPICE may be skipped:", _e)
    HAVE_COCOCAP = False

try:
    from bert_score import score as bert_score
    USE_BERTSCORE = True
except Exception:
    USE_BERTSCORE = False

# ---- Pull training constants if not already defined ----
DEVICE              = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE          = globals().get("BATCH_SIZE", 4)
GRAD_ACCUM_STEPS    = globals().get("GRAD_ACCUM_STEPS", 4)
LEARNING_RATE       = globals().get("LEARNING_RATE", 2e-4)
NUM_EPOCHS          = globals().get("NUM_EPOCHS", 3)    
MAX_CAPTION_LENGTH  = globals().get("MAX_CAPTION_LENGTH", 96)
PAL_WARMUP_STEPS    = globals().get("PAL_WARMUP_STEPS", 2000)
TOP_N               = globals().get("TOP_N", 5)          # beams / return sequences
MAXVIT_MODEL_NAME   = globals().get("MAXVIT_MODEL_NAME", "maxvit_base_tf_224.in1k")
MBART_MODEL_NAME    = globals().get("MBART_MODEL_NAME", "facebook/mbart-large-50-many-to-many-mmt")

# ---- Recommended: turn OFF InfoNCE/OT for PAL ablation ----
USE_INFO_NCE = False
USE_OT       = False

# ---- Helper: reproducibility ----
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

# ---- Safe image loader ----
def safe_load_image(p):
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

# ---- Tokenizer-based tokenization (for BLEU) ----
def tokenize_bn_model(sentences, tokenizer):
    toks = []
    for s in sentences:
        s = (s or "").strip()
        if not s: 
            continue
        ids = tokenizer(s, return_tensors="pt")["input_ids"][0]
        toks.append(tokenizer.convert_ids_to_tokens(ids))
    return toks

# ---- Build df_val dataloader for speed (images may be reused) ----
assert 'df_val' in globals() and not df_val.empty, "df_val must exist and be non-empty for ablation."
assert 'BengaliCaptionDataset' in globals(), "Dataset class not found."
assert 'collate_fn' in globals(), "collate_fn not found."
assert 'train_transform_safe' in globals(), "train_transform_safe not found."
assert 'eval_transform_safe' in globals(), "eval_transform_safe not found."

# We still train on df_train; eval on df_val with generation.
ds_train = BengaliCaptionDataset(df_train, train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                      pin_memory=True, collate_fn=collate_fn)

# ---- Key metric for ranking/plots ----
KEY_METRIC = "CIDEr"  # fallback to BLEU-4 if COCOCAP missing

# ---- Ablation grids ----
PATCH_WEIGHTS = [0.1, 0.3, 0.5, 0.8]
ATTN_TEMPS    = [0.7, 1.0, 1.3]
TOPK_RATIOS   = [0.1, 0.3, 0.5]

# ---- Metric objects ----
chencherry = SmoothingFunction()
if HAVE_COCOCAP:
    meteor_scorer = Meteor()
    rouge_scorer  = Rouge()
    cider_scorer  = Cider()
    spice_scorer  = Spice()

# ---- Modified training wrapper to freeze topk (no late schedule during ablation) ----
def train_for_epochs_ablation(
    model, dataloader, optimizer, scheduler, scaler, device,
    grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=NUM_EPOCHS,
    base_pal_weight=0.3, pal_warmup_steps=PAL_WARMUP_STEPS,
    freeze_topk=True
):
    # default unfreezing schedule (same as your code)
    model.set_trainable_decoder_layers(last_k=2)
    global_step = 0
    for epoch in range(num_epochs):
        # progressive unfreeze
        if epoch == max(0, num_epochs//3):   # epoch 1 if num_epochs=3; epoch 2 if 7, etc.
            model.set_trainable_decoder_layers(last_k=6)
        if epoch == max(0, (2*num_epochs)//3):
            model.set_trainable_decoder_layers(last_k=12)

        # DO NOT override topk if freezing; respect ablation grid value
        if not freeze_topk:
            if epoch >= 3: model.topk_ratio = 0.5
            if epoch >= 5: model.topk_ratio = 0.25

        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_epoch_loss = 0.0; steps = 0

        for batch in DataLoader(dataloader.dataset, batch_size=dataloader.batch_size, shuffle=True,
                                num_workers=0, pin_memory=True, collate_fn=collate_fn):
            if batch is None: 
                continue

            # PAL warmup on the fly
            pal_scale = min(1.0, float(global_step) / max(1, pal_warmup_steps))
            model.patch_alignment_weight = float(base_pal_weight) * pal_scale

            real_pixel_values      = batch["real_pixel_values"].to(device, non_blocking=True)
            synthetic_pixel_values = batch["synthetic_pixel_values"].to(device, non_blocking=True)
            labels                 = batch["labels"].to(device, non_blocking=True)
            decoder_attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device=="cuda")):
                outputs = model(
                    real_pixel_values=real_pixel_values,
                    synthetic_pixel_values=synthetic_pixel_values,
                    labels=labels,
                    attention_mask=decoder_attention_mask
                )
                loss = outputs["loss"] / grad_accum_steps

            scaler.scale(loss).backward()
            if (steps + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            total_epoch_loss += float(outputs["loss"].detach())
            steps += 1; global_step += 1

            # free
            del real_pixel_values, synthetic_pixel_values, labels, decoder_attention_mask, outputs
            if device == "cuda":
                torch.cuda.empty_cache()

        print(f"  · Epoch {epoch+1}/{num_epochs}: Avg Loss = {total_epoch_loss/max(1,steps):.4f}")

# ---- Evaluation on df_val (best-of-N) ----
def evaluate_on_df_val(model, tokenizer, df_val, eval_transform, beams=TOP_N):
    results = []
    for idx, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Eval df_val"):
        img_path = row["real_image_path"]
        refs_bn  = [row["caption_bn"]]
        img = safe_load_image(img_path)
        if img is None:
            continue

        with torch.no_grad():
            with torch.amp.autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"),
                                    dtype=(torch.float16 if DEVICE=="cuda" else torch.float32)):
                img_t = eval_transform(img).unsqueeze(0).to(DEVICE)
                gens = model.generate(
                    real_pixel_values=img_t,
                    tokenizer=tokenizer,
                    max_new_tokens=128,
                    num_beams=beams,
                    num_return_sequences=beams,
                    length_penalty=1.3,
                    no_repeat_ngram_size=3,
                    min_new_tokens=16,
                    repetition_penalty=1.02
                )

        # Tokenize refs once (for BLEU)
        tokenized_refs = tokenize_bn_model(refs_bn, tokenizer)

        best_caption = gens[0]
        best = {"BLEU-1": -1, "BLEU-2": -1, "BLEU-3": -1, "BLEU-4": -1,
                "METEOR": -1, "CIDEr": -1, "ROUGE-L": -1, "SPICE": -1}
        if USE_BERTSCORE: best["BERTScore-F1"] = -1.0

        for cand in gens:
            tokenized_pred = tokenize_bn_model([cand], tokenizer)[0]
            # BLEU 1..4
            weights = [
                (1,0,0,0),
                (0.5,0.5,0,0),
                (1/3,1/3,1/3,0),
                (0.25,0.25,0.25,0.25),
            ]
            try: b1 = corpus_bleu([tokenized_refs],[tokenized_pred],weights=weights[0],smoothing_function=chencherry.method1)*100
            except: b1 = 0.0
            try: b2 = corpus_bleu([tokenized_refs],[tokenized_pred],weights=weights[1],smoothing_function=chencherry.method1)*100
            except: b2 = 0.0
            try: b3 = corpus_bleu([tokenized_refs],[tokenized_pred],weights=weights[2],smoothing_function=chencherry.method1)*100
            except: b3 = 0.0
            try: b4 = corpus_bleu([tokenized_refs],[tokenized_pred],weights=weights[3],smoothing_function=chencherry.method1)*100
            except: b4 = 0.0

            m=c=r=sp=0.0
            if HAVE_COCOCAP:
                try:
                    refs = {idx: refs_bn}; hyps = {idx: [cand]}
                    m = float(Meteor().compute_score(refs, hyps)[0])
                    c = float(Cider().compute_score(refs, hyps)[0])
                    r = float(Rouge().compute_score(refs, hyps)[0])
                    sp = float(Spice().compute_score(refs, hyps)[0])
                except Exception:
                    pass

            bf1 = 0.0
            if USE_BERTSCORE:
                try:
                    _, _, F1 = bert_score([cand]*len(refs_bn), refs_bn, lang="bn", rescale_with_baseline=False)
                    bf1 = float(F1.mean())
                except Exception:
                    bf1 = 0.0

            metrics = {"BLEU-1": b1, "BLEU-2": b2, "BLEU-3": b3, "BLEU-4": b4,
                       "METEOR": m, "CIDEr": c, "ROUGE-L": r, "SPICE": sp,
                       "BERTScore-F1": bf1 if USE_BERTSCORE else 0.0}
            if sum(metrics.values()) > sum(best.values()):
                best, best_caption = metrics, cand

        results.append(best)

    # Average metrics
    if not results:
        return {k: 0.0 for k in ["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR","CIDEr","ROUGE-L","SPICE"] + (["BERTScore-F1"] if USE_BERTSCORE else [])}
    dfm = pd.DataFrame(results)
    return {k: float(dfm[k].mean()) for k in dfm.columns}

# ---- Ablation main loop ----
all_rows = []
start_all = time.time()
run_id = 0

for patch_w, temp, topk in itertools.product(PATCH_WEIGHTS, ATTN_TEMPS, TOPK_RATIOS):
    run_id += 1
    set_seed(42 + run_id)
    print(f"\n=== PAL Ablation Run {run_id} / {len(PATCH_WEIGHTS)*len(ATTN_TEMPS)*len(TOPK_RATIOS)} ===")
    print(f"  patch_alignment_weight={patch_w} | attn_temp={temp} | topk_ratio={topk}")

    # Build fresh model for each run
    model = MaxVitMbartCaptioningModel(
        MAXVIT_MODEL_NAME, MBART_MODEL_NAME,
        bn_in_token_id=tokenizer.lang_code_to_id.get("bn_IN"),
        patch_alignment_weight=0.0,           # will be warmed via base_pal_weight
        attn_temp=float(temp),
        topk_ratio=float(topk),
        use_last_k_cross_layers=globals().get("PAL_CFG", {}).get("use_last_k_cross_layers", 2),
        pool_factor=globals().get("PAL_CFG", {}).get("pool_factor", 2),
        multi_scale=globals().get("PAL_CFG", {}).get("multi_scale", True),
        use_info_nce=False,
        info_nce_temp=globals().get("INFO_NCE_CFG", {}).get("temp", 0.07),
        info_nce_coef=globals().get("INFO_NCE_CFG", {}).get("coef", 0.3),
        use_ot=False,
        ot_reg=globals().get("OT_CFG", {}).get("reg", 0.05),
        ot_iters=globals().get("OT_CFG", {}).get("iters", 30),
        ot_topk_ratio=globals().get("OT_CFG", {}).get("topk_ratio", 0.10),
        ot_coef=globals().get("OT_CFG", {}).get("coef", 0.5)
    ).to(DEVICE)

    # Optimizers/scheduler/scaler
    t_total = max(1, len(dl_train)) * NUM_EPOCHS // max(1, GRAD_ACCUM_STEPS)
    warmup_steps = int(0.06 * t_total)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=LEARNING_RATE, weight_decay=0.01)
    from transformers import get_linear_schedule_with_warmup
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, t_total)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE=="cuda"))

    # Train
    train_for_epochs_ablation(
        model, dl_train, optimizer, scheduler, scaler, DEVICE,
        grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=NUM_EPOCHS,
        base_pal_weight=patch_w, pal_warmup_steps=PAL_WARMUP_STEPS, freeze_topk=True
    )

    # Evaluate on df_val
    metrics = evaluate_on_df_val(model, tokenizer, df_val, eval_transform_safe, beams=TOP_N)

    row = {
        "run_id": run_id,
        "patch_alignment_weight": patch_w,
        "attn_temp": temp,
        "topk_ratio": topk,
        **metrics
    }
    all_rows.append(row)

    # cleanup
    del model, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ---- Save results ----
ablation_df = pd.DataFrame(all_rows)
ABLATION_CSV = "/kaggle/working/pal_ablation_results.csv"
ablation_df.to_csv(ABLATION_CSV, index=False)
print(f"\n Saved ablation results to: {ABLATION_CSV}")

# ---- Pick best by KEY_METRIC (fallback BLEU-4 if missing) ----
metric_for_rank = KEY_METRIC if KEY_METRIC in ablation_df.columns else "BLEU-4"
best_row = ablation_df.sort_values(metric_for_rank, ascending=False).iloc[0]
print("\nBest configuration (by {}):".format(metric_for_rank))
print(best_row.to_dict())

# ================================
# Simple plots (bars) aggregating each hyperparam vs KEY_METRIC
# ================================
def _agg_and_plot(df, col, metric=metric_for_rank, out_png_path=None):
    agg = df.groupby(col)[metric].mean().reset_index().sort_values(col)
    plt.figure(figsize=(6,4))
    plt.bar([str(x) for x in agg[col].values], agg[metric].values)
    plt.xlabel(col); plt.ylabel(f"Avg {metric}"); plt.title(f"{col} vs {metric} (mean over others)")
    plt.tight_layout()
    if out_png_path:
        plt.savefig(out_png_path, dpi=200)
    plt.show()
    return agg

PLOT1 = "/kaggle/working/ablate_patch_weight_vs_{}.png".format(metric_for_rank)
PLOT2 = "/kaggle/working/ablate_attn_temp_vs_{}.png".format(metric_for_rank)
PLOT3 = "/kaggle/working/ablate_topk_ratio_vs_{}.png".format(metric_for_rank)

print("\nCreating summary plots...")
agg_pw = _agg_and_plot(ablation_df, "patch_alignment_weight", metric_for_rank, PLOT1)
agg_at = _agg_and_plot(ablation_df, "attn_temp",               metric_for_rank, PLOT2)
agg_tk = _agg_and_plot(ablation_df, "topk_ratio",              metric_for_rank, PLOT3)

print(f"Saved:\n  {PLOT1}\n  {PLOT2}\n  {PLOT3}")
